# SniffTest Train Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skrript/sniff-test/blob/main/snifftest_train.ipynb)

This notebook is the self-contained training entrypoint for SniffTest. It clones the repo, installs the training stack, starts the environment server from repo code, optionally regenerates SFT trajectories, runs SFT and GRPO, evaluates baseline vs post-SFT vs post-RL, and renders the training curves inline.


## Runtime Notes

- Default install path in notebooks is `%pip install -e ".[train]"`, not `uv sync --extra train`.
- Secrets can come from notebook/cloud secrets or from a local `.env` file in the cloned repo.
- `SNIFFTEST_MODEL_NAME` is the shared env var for both training and inference. `MODEL_NAME` remains a fallback for compatibility.
- If `HF_TOKEN` or `HUGGINGFACE_HUB_TOKEN` is missing, the notebook falls back to `huggingface_hub.notebook_login()`.
- Runtime artifacts are written under `temp/snifftest_train/` by default and should stay uncommitted.


In [ ]:
from __future__ import annotations

import importlib.util
import os
import shlex
import socket
import subprocess
import sys
import time
from pathlib import Path
from urllib.parse import urlparse

DEFAULT_REPO_URL = "https://github.com/skrript/sniff-test.git"
DEFAULT_MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"

WORKSPACE = Path.cwd().resolve()
DEFAULT_REPO_DIR = (
    WORKSPACE
    if (WORKSPACE / "pyproject.toml").exists() and (WORKSPACE / "snifftest_train.ipynb").exists()
    else WORKSPACE / "sniff-test"
)
REPO_URL = os.getenv("SNIFFTEST_REPO_URL", DEFAULT_REPO_URL)
REPO_DIR = Path(os.getenv("SNIFFTEST_REPO_DIR", str(DEFAULT_REPO_DIR))).resolve()
ARTIFACT_DIR_NAME = os.getenv("SNIFFTEST_ARTIFACT_DIR", "temp/snifftest_train")
BASE_URL = os.getenv("SNIFFTEST_BASE_URL", "http://127.0.0.1:8000")
MODEL_NAME = os.getenv("SNIFFTEST_MODEL_NAME") or os.getenv("MODEL_NAME") or DEFAULT_MODEL_NAME
OPENAI_SFT_MODEL = os.getenv("OPENAI_SFT_MODEL", "gpt-5-mini")
REGENERATE_SFT_DATA = os.getenv("REGENERATE_SFT_DATA", "0").strip().lower() in {"1", "true", "yes", "on"}
USE_VLLM = os.getenv("SNIFFTEST_USE_VLLM", "0").strip().lower() in {"1", "true", "yes", "on"}

TRAIN_REQUIREMENTS = [
    "python-dotenv>=1.0.0",
    "accelerate>=1.13.0",
    "datasets>=3.6.0",
    "matplotlib>=3.8.0",
    "peft>=0.19.1",
    "trl>=0.17.0",
    "unsloth>=2025.9.5",
]
if USE_VLLM:
    TRAIN_REQUIREMENTS.append("vllm>=0.8.0")


def run_command(args: list[str], cwd: Path | None = None) -> None:
    print("$", shlex.join(args))
    subprocess.run(args, cwd=cwd, check=True)


def pip_install(args: list[str], cwd: Path | None = None) -> None:
    run_command([sys.executable, "-m", "pip", "install", *args], cwd=cwd)


def pip_uninstall(args: list[str], cwd: Path | None = None) -> None:
    run_command([sys.executable, "-m", "pip", "uninstall", "-y", *args], cwd=cwd)


if REPO_DIR.exists():
    print(f"Using existing repo checkout: {REPO_DIR}")
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run_command(["git", "clone", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
pip_install(["--upgrade", "pip", "setuptools", "wheel"])
pip_install(["-e", "."])
pip_install(TRAIN_REQUIREMENTS)
if not USE_VLLM and importlib.util.find_spec("vllm") is not None:
    print("SNIFFTEST_USE_VLLM is disabled; uninstalling vllm to avoid TRL import-time incompatibilities.")
    pip_uninstall(["vllm"])

HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_HUB_TOKEN")
if not HF_TOKEN:
    from huggingface_hub import notebook_login

    notebook_login()
    HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_HUB_TOKEN")
if HF_TOKEN:
    os.environ.setdefault("HF_TOKEN", HF_TOKEN)
    os.environ.setdefault("HUGGINGFACE_HUB_TOKEN", HF_TOKEN)

ARTIFACT_DIR = (REPO_DIR / ARTIFACT_DIR_NAME).resolve()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo dir: {REPO_DIR}")
print(f"Artifacts: {ARTIFACT_DIR}")
print(f"Model: {MODEL_NAME}")
print(f"HF token configured: {bool(HF_TOKEN)}")
print(f"Regenerate SFT trajectories: {REGENERATE_SFT_DATA}")
print(f"Use vLLM: {USE_VLLM}")
print(f"Base URL: {BASE_URL}")


In [ ]:
from __future__ import annotations

import atexit
import asyncio
import importlib.util
import json
import os
import signal
import sys
import textwrap
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from IPython.display import HTML, Image, display

ROOT = REPO_DIR
DATA_DIR = ROOT / "data"
SFT_TRAJECTORIES_PATH = DATA_DIR / "sft_trajectories.jsonl"
SFT_SCENARIOS_PATH = DATA_DIR / "sft_scenarios.json"
CLAIMS_DATASET_PATH = DATA_DIR / "claims_dataset.json"
TEST_DATASET_PATH = DATA_DIR / "test_dataset.json"
SFT_CHAT_PATH = ARTIFACT_DIR / "sft_chat_examples.jsonl"
SFT_OUTPUT_DIR = ARTIFACT_DIR / "sft_warm_start"
TRAIN_OUTPUT_DIR = ARTIFACT_DIR / "training_outputs"
FINAL_DIR = TRAIN_OUTPUT_DIR / "final_merged"
ADAPTER_DIR = TRAIN_OUTPUT_DIR / "lora_adapters"
REWARD_HISTORY_PATH = TRAIN_OUTPUT_DIR / "reward_history.json"
EVAL_METRICS_PATH = TRAIN_OUTPUT_DIR / "test_eval_metrics.json"
REPORT_IMAGE_PATH = TRAIN_OUTPUT_DIR / "training_and_eval_report.png"

load_dotenv(ROOT / ".env", override=False)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    from snifftest_env.client import SniffTestEnv
    from snifftest_env.models import InvestigateAction
    from snifftest_env.server.snifftest_environment import SniffTestEnvironment
except ImportError:
    from client import SniffTestEnv
    from models import InvestigateAction
    from server.snifftest_environment import SniffTestEnvironment

SERVER_PROCESS: subprocess.Popen[str] | None = None


def _port_from_base_url(base_url: str) -> int:
    parsed = urlparse(base_url)
    if parsed.port is not None:
        return parsed.port
    return 443 if parsed.scheme == "https" else 80


SERVER_PORT = _port_from_base_url(BASE_URL)


def wait_for_port(host: str, port: int, timeout: float = 30.0) -> None:
    start = time.time()
    while time.time() - start < timeout:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.settimeout(1.0)
            if sock.connect_ex((host, port)) == 0:
                return
        time.sleep(0.25)
    raise TimeoutError(f"Timed out waiting for {host}:{port}")


def start_server(base_url: str = BASE_URL) -> subprocess.Popen[str]:
    global SERVER_PROCESS
    if SERVER_PROCESS is not None and SERVER_PROCESS.poll() is None:
        print(f"Server already running on {base_url}")
        return SERVER_PROCESS

    env = os.environ.copy()
    env["PYTHONPATH"] = str(ROOT)
    SERVER_PROCESS = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "uvicorn",
            "server.app:app",
            "--host",
            "127.0.0.1",
            "--port",
            str(SERVER_PORT),
        ],
        cwd=ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    wait_for_port("127.0.0.1", SERVER_PORT)
    print(f"Server started at {base_url} with pid {SERVER_PROCESS.pid}")
    return SERVER_PROCESS


def stop_server() -> None:
    global SERVER_PROCESS
    if SERVER_PROCESS is None:
        return
    if SERVER_PROCESS.poll() is None:
        SERVER_PROCESS.send_signal(signal.SIGTERM)
        try:
            SERVER_PROCESS.wait(timeout=10)
        except subprocess.TimeoutExpired:
            SERVER_PROCESS.kill()
    SERVER_PROCESS = None


atexit.register(stop_server)
start_server()


In [ ]:
async def server_smoke_test() -> None:
    async with SniffTestEnv(base_url=BASE_URL) as env:
        reset_result = await env.reset(task_level="easy")
        obs = reset_result.observation
        print(f"Claim: {obs.claim}")
        print(f"Visible sources: {len(obs.available_sources)}")
        print(f"Steps remaining: {obs.steps_remaining}")


await server_smoke_test()


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import torch
import unsloth

MODEL_NAME = os.getenv("SNIFFTEST_MODEL_NAME") or os.getenv("MODEL_NAME") or DEFAULT_MODEL_NAME
MAX_SEQ_LENGTH = int(os.getenv("SNIFFTEST_MAX_SEQ_LENGTH", "2048"))
LORA_RANK = int(os.getenv("SNIFFTEST_LORA_RANK", "16"))
MAX_TURNS = 10
USE_VLLM = os.getenv("SNIFFTEST_USE_VLLM", "1").strip().lower() in {"1", "true", "yes", "on"}
USE_UNSLOTH_FAST_INFERENCE = os.getenv("SNIFFTEST_FAST_INFERENCE", "0").strip().lower() in {"1", "true", "yes", "on"}
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = not USE_BF16
VLLM_MODE = os.getenv("SNIFFTEST_VLLM_MODE", "colocate")
VLLM_GPU_MEMORY_UTILIZATION = float(os.getenv("SNIFFTEST_VLLM_GPU_MEMORY_UTILIZATION", "0.3"))
MAX_PROMPT_LENGTH = int(os.getenv("SNIFFTEST_MAX_PROMPT_LENGTH", "2048"))
MAX_COMPLETION_LENGTH = int(os.getenv("SNIFFTEST_MAX_COMPLETION_LENGTH", "2048"))
NUM_GENERATIONS = int(os.getenv("SNIFFTEST_NUM_GENERATIONS", "2"))
TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]
CURRICULUM = [
    {"task_level": "easy", "episodes": 300, "reward_threshold": 0.45},
    {"task_level": "medium", "episodes": 300, "reward_threshold": 0.30},
    {"task_level": "hard", "episodes": 200, "reward_threshold": None},
]
ACTION_FIELDS_BY_TYPE = {
    "search": {"query"},
    "open_source": {"source_id"},
    "cross_reference": {"source_ids"},
    "trace_origin": {"source_id"},
    "check_metadata": {"source_id"},
    "submit_verdict": {"verdict", "confidence", "justification"},
}
VALID_VERDICTS = {"true", "false", "misleading", "unverifiable"}
SYSTEM_PROMPT = textwrap.dedent(
    """
    You are an expert fact-checker investigating claims for accuracy.

    Available actions (return as JSON, one action per response):
    {"action_type": "search", "query": "your search query"}
    {"action_type": "open_source", "source_id": "src_xxx"}
    {"action_type": "cross_reference", "source_ids": ["src_xxx", "src_yyy"]}
    {"action_type": "trace_origin", "source_id": "src_xxx"}
    {"action_type": "check_metadata", "source_id": "src_xxx"}
    {"action_type": "submit_verdict", "verdict": "true|false|misleading|unverifiable",
     "confidence": 0.0-1.0, "justification": "reasoning citing specific source IDs and domains"}

    Investigation strategy:
    1. Search with a targeted query to discover sources
    2. Open the most relevant sources to read full content
    3. Cross-reference conflicting sources
    4. Check metadata on suspicious sources
    5. Submit verdict with detailed justification

    Return ONLY the JSON action. No other text.
    """
).strip()
SFT_GENERATION_PROMPT = textwrap.dedent(
    """
    You are an expert fact-checker demonstrating ideal investigation technique.
    Given a claim and three visible sources, produce a precise step-by-step investigation.
    Return ONLY a JSON array of actions. Each action must match exactly one of these schemas:
    {"action_type": "search", "query": "string"}
    {"action_type": "open_source", "source_id": "string"}
    {"action_type": "cross_reference", "source_ids": ["string", "string"]}
    {"action_type": "trace_origin", "source_id": "string"}
    {"action_type": "check_metadata", "source_id": "string"}
    {"action_type": "submit_verdict", "verdict": "true|false|misleading|unverifiable",
     "confidence": 0.0, "justification": "detailed reasoning citing specific sources"}
    Constraints:
    - Use only the listed source IDs.
    - Produce 4-6 actions total.
    - Start with search.
    - End with submit_verdict.
    - Prefer trajectories that teach clean JSON formatting and disciplined tool use.
    No preamble. No markdown. Only the JSON array.
    """
).strip()


def load_unsloth_model(
    model_name: str = MODEL_NAME,
    max_seq_length: int = MAX_SEQ_LENGTH,
    lora_rank: int = LORA_RANK,
    add_lora: bool = True,
    fast_inference: bool = USE_UNSLOTH_FAST_INFERENCE,
):
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq_length,
        load_in_4bit=True,
        fast_inference=fast_inference,
    )
    if add_lora:
        model = FastLanguageModel.get_peft_model(
            model,
            r=lora_rank,
            target_modules=TARGET_MODULES,
            lora_alpha=lora_rank * 2,
            lora_dropout=0,
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=42,
        )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer


def load_json(path: Path) -> Any:
    return json.loads(path.read_text())


def load_jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open() as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_json(path: Path, payload: Any) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2))
    return path


def format_available_sources(visible: list[dict[str, Any]]) -> str:
    return "\n".join(
        f"- [{src['source_id']}] {src['title']} ({src['domain']}): {src['snippet']}"
        for src in visible
    )


def make_sft_user_turn(record: dict[str, Any], prior_actions: list[dict[str, Any]]) -> str:
    lines = [
        f"Investigate this claim: {record['claim']}",
        "",
        "Available sources:",
        format_available_sources(record.get("visible_sources", [])),
    ]
    if prior_actions:
        lines.extend(
            [
                "",
                "Previous expert actions:",
                *[json.dumps(action, ensure_ascii=True, separators=(",", ":")) for action in prior_actions],
            ]
        )
    lines.extend(["", "Return the next best single JSON action."])
    return "\n".join(lines)


def load_sft_trajectories(path: Path = SFT_TRAJECTORIES_PATH) -> list[dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"{path} not found.")
    return load_jsonl(path)


def build_sft_examples(records: list[dict[str, Any]]) -> list[dict[str, Any]]:
    examples: list[dict[str, Any]] = []
    for record in records:
        visible_sources = record.get("visible_sources")
        if not isinstance(visible_sources, list) or len(visible_sources) < 3:
            continue
        actions = record.get("actions", [])
        for idx, action in enumerate(actions):
            examples.append(
                {
                    "scenario_id": record["scenario_id"],
                    "difficulty": record["difficulty"],
                    "step_index": idx,
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": make_sft_user_turn(record, actions[:idx])},
                        {"role": "assistant", "content": json.dumps(action, ensure_ascii=True, separators=(",", ":"))},
                    ],
                }
            )
    return examples


def export_sft_examples(output_path: Path = SFT_CHAT_PATH, input_path: Path = SFT_TRAJECTORIES_PATH) -> Path:
    examples = build_sft_examples(load_sft_trajectories(input_path))
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w") as handle:
        for example in examples:
            handle.write(json.dumps(example) + "\n")
    print(f"Exported {len(examples)} SFT examples -> {output_path}")
    return output_path


def load_sft_scenarios(path: Path = SFT_SCENARIOS_PATH) -> list[dict[str, Any]]:
    scenarios = load_json(path)
    if not isinstance(scenarios, list):
        raise ValueError("SFT scenarios must be a top-level JSON array.")
    return scenarios


def prompt_for_scenario(scenario: dict[str, Any]) -> str:
    visible_sources = "\n".join(
        f"- [{source['source_id']}] {source['title']} ({source['domain']}): {source['snippet']}"
        for source in scenario["sources"][:3]
    )
    teaching_focus = ", ".join(scenario.get("teaching_focus", [])) or "clean action formatting"
    return f"""Claim: {scenario['claim']}

Available sources:
{visible_sources}

Ground truth (for your reference only, not visible to agent):
- Truth label: {scenario['truth_label']}
- Teaching focus: {teaching_focus}
- Scenario notes: {scenario.get('notes', '')}

Produce an ideal investigation sequence that:
1. Starts with a targeted search.
2. Opens decisive sources from the visible list.
3. Uses cross_reference, metadata, or trace only when it improves the investigation.
4. Ends with a well-reasoned verdict citing concrete source IDs."""


def validate_actions(scenario: dict[str, Any], actions: list[dict[str, Any]]) -> None:
    if not isinstance(actions, list) or not actions:
        raise ValueError(f"{scenario['scenario_id']}: generation did not return a non-empty action list.")
    valid_source_ids = {source["source_id"] for source in scenario["sources"][:3]}
    for idx, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(f"{scenario['scenario_id']}: action {idx} is not an object.")
        action_type = action.get("action_type")
        if action_type not in ACTION_FIELDS_BY_TYPE:
            raise ValueError(f"{scenario['scenario_id']}: action {idx} has invalid action_type {action_type!r}.")
        missing_fields = sorted(field for field in ACTION_FIELDS_BY_TYPE[action_type] if field not in action)
        if missing_fields:
            raise ValueError(f"{scenario['scenario_id']}: action {idx} missing required fields {missing_fields}.")
        if action_type == "search":
            if not isinstance(action["query"], str) or not action["query"].strip():
                raise ValueError(f"{scenario['scenario_id']}: invalid search query.")
        elif action_type in {"open_source", "trace_origin", "check_metadata"}:
            if action["source_id"] not in valid_source_ids:
                raise ValueError(f"{scenario['scenario_id']}: non-visible source_id {action['source_id']}.")
        elif action_type == "cross_reference":
            source_ids = action["source_ids"]
            if not isinstance(source_ids, list) or len(source_ids) != 2 or len(set(source_ids)) != 2:
                raise ValueError(f"{scenario['scenario_id']}: cross_reference must use two distinct source IDs.")
            if any(source_id not in valid_source_ids for source_id in source_ids):
                raise ValueError(f"{scenario['scenario_id']}: cross_reference uses a non-visible source.")
        elif action_type == "submit_verdict":
            if action["verdict"] not in VALID_VERDICTS:
                raise ValueError(f"{scenario['scenario_id']}: invalid verdict.")
            confidence = action["confidence"]
            if not isinstance(confidence, (int, float)) or not 0.0 <= float(confidence) <= 1.0:
                raise ValueError(f"{scenario['scenario_id']}: invalid confidence.")
            if not isinstance(action["justification"], str) or not action["justification"].strip():
                raise ValueError(f"{scenario['scenario_id']}: empty justification.")
    if actions[0].get("action_type") != "search":
        raise ValueError(f"{scenario['scenario_id']}: first action must be search.")
    if actions[-1].get("action_type") != "submit_verdict":
        raise ValueError(f"{scenario['scenario_id']}: final action must be submit_verdict.")
    if actions[-1].get("verdict") != scenario["truth_label"]:
        raise ValueError(f"{scenario['scenario_id']}: final verdict must match truth_label.")


def visible_sources_for_scenario(scenario: dict[str, Any]) -> list[dict[str, str]]:
    return [
        {
            "source_id": source["source_id"],
            "title": source["title"],
            "domain": source["domain"],
            "snippet": source["snippet"],
        }
        for source in scenario["sources"][:3]
    ]


def generate_sft_trajectories(input_path: Path = SFT_SCENARIOS_PATH, output_path: Path = SFT_TRAJECTORIES_PATH) -> Path:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY is required when REGENERATE_SFT_DATA=1.")
    from openai import OpenAI

    client = OpenAI()
    scenarios = load_sft_scenarios(input_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    written = 0
    with output_path.open("w") as out_f:
        for scenario in scenarios:
            response = client.responses.create(
                model=OPENAI_SFT_MODEL,
                input=[
                    {"role": "system", "content": SFT_GENERATION_PROMPT},
                    {"role": "user", "content": prompt_for_scenario(scenario)},
                ],
            )
            actions = json.loads(response.output_text)
            validate_actions(scenario, actions)
            record = {
                "scenario_id": scenario["scenario_id"],
                "difficulty": scenario["difficulty"],
                "claim": scenario["claim"],
                "visible_sources": visible_sources_for_scenario(scenario),
                "actions": actions,
                "truth_label": scenario["truth_label"],
            }
            out_f.write(json.dumps(record, ensure_ascii=True) + "\n")
            written += 1
    print(f"Generated {written} trajectories -> {output_path}")
    return output_path


def train_sft(output_dir: Path = SFT_OUTPUT_DIR, epochs: int = 2, learning_rate: float = 2e-4, model_name: str = MODEL_NAME):
    from datasets import Dataset
    from trl import SFTConfig, SFTTrainer

    model, tokenizer = load_unsloth_model(model_name=model_name)
    examples = build_sft_examples(load_sft_trajectories())
    rendered = []
    for example in examples:
        rendered.append(
            {
                "text": tokenizer.apply_chat_template(
                    example["messages"],
                    tokenize=False,
                    add_generation_prompt=False,
                )
            }
        )
    dataset = Dataset.from_list(rendered)
    output_dir.mkdir(parents=True, exist_ok=True)
    trainer = SFTTrainer(
        model=model,
        args=SFTConfig(
            output_dir=str(output_dir),
            num_train_epochs=epochs,
            learning_rate=learning_rate,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            logging_steps=10,
            save_steps=50,
            bf16=USE_BF16,
            fp16=USE_FP16,
            max_seq_length=MAX_SEQ_LENGTH,
            dataset_text_field="text",
            report_to="none",
        ),
        train_dataset=dataset,
        processing_class=tokenizer,
    )
    trainer.train()
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    print(f"SFT warm start saved -> {output_dir}")
    return model, tokenizer, trainer


def obs_to_text(obs: Any, step: int) -> str:
    lines = [
        f"Step {step}/{MAX_TURNS}  |  Steps remaining: {obs.steps_remaining}",
        "",
        f"CLAIM: {obs.claim}",
        "",
        f"AVAILABLE SOURCES ({len(obs.available_sources)} discovered):",
    ]
    for source in obs.available_sources:
        tag = " [READ]" if source.retrieved else ""
        lines.append(f"  [{source.source_id}] {source.title} ({source.domain}){tag}")
        lines.append(f"    {source.snippet}")
    if obs.opened_content:
        lines.extend(["", "LAST OPENED:", obs.opened_content[:500]])
    if obs.cross_reference_result:
        lines.extend(["", "CROSS-REFERENCE:", obs.cross_reference_result[:300]])
    if obs.trace_result:
        lines.extend(["", "TRACE RESULT:", obs.trace_result[:300]])
    if obs.metadata_result:
        lines.extend(["", "METADATA CHECK:", obs.metadata_result[:300]])
    if obs.action_history:
        lines.extend(["", "ACTION HISTORY (last 5):"])
        for history in obs.action_history[-5:]:
            lines.append(f"  Step {history.step} [{history.action_type}]: {history.result_summary}")
    if obs.message:
        lines.extend(["", f"FEEDBACK: {obs.message}"])
    lines.extend(["", "Next action (JSON only):"])
    return "\n".join(lines)


def parse_action(text: str) -> tuple[dict[str, Any], bool]:
    text = (text or "").strip()
    if text.startswith("```"):
        parts = text.split("```")
        text = parts[1] if len(parts) > 1 else text
        if text.startswith("json"):
            text = text[4:]
    text = text.strip()
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("{"):
            try:
                return json.loads(line), True
            except json.JSONDecodeError:
                continue
    try:
        return json.loads(text), True
    except json.JSONDecodeError:
        return {"action_type": "search", "query": "claim evidence"}, False


def run_episode(model: Any, tokenizer: Any, task_level: str, env: Any, scenario_id: str | None = None) -> dict[str, Any]:
    import torch

    obs = env.reset(task_level=task_level, scenario_id=scenario_id)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Investigate this claim: {obs.claim}\n\nAvailable sources:\n"
                + "\n".join(
                    f"- [{source.source_id}] {source.title} ({source.domain}): {source.snippet}"
                    for source in obs.available_sources
                )
            ),
        },
    ]
    prompts: list[str] = []
    completions: list[str] = []
    rewards: list[float] = []
    done = False
    total_reward = 0.0
    invalid_output_penalty = 0.0
    valid_json_steps = 0
    used_advanced_tool = False
    final_verdict = None
    for _ in range(MAX_TURNS):
        if done:
            break
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        completion = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()
        try:
            action_dict, valid = parse_action(completion)
            if not valid:
                reward = -0.1
                rewards.append(reward)
                total_reward += reward
                invalid_output_penalty += reward
                prompts.append(prompt)
                completions.append(completion)
                messages.append({"role": "assistant", "content": completion})
                messages.append({"role": "user", "content": "Invalid action format. Provide one valid JSON action."})
                continue
            action = InvestigateAction(**action_dict)
        except Exception:
            reward = -0.1
            rewards.append(reward)
            total_reward += reward
            invalid_output_penalty += reward
            prompts.append(prompt)
            completions.append(completion)
            messages.append({"role": "assistant", "content": completion})
            messages.append({"role": "user", "content": "Invalid action schema. Provide one valid JSON action."})
            continue
        valid_json_steps += 1
        if action.action_type in {"cross_reference", "trace_origin", "check_metadata"}:
            used_advanced_tool = True
        if action.action_type == "submit_verdict":
            final_verdict = action.verdict
        obs = env.step(action)
        reward = float(obs.reward or 0.0)
        done = bool(obs.done)
        total_reward += reward
        prompts.append(prompt)
        completions.append(completion)
        rewards.append(reward)
        messages.append({"role": "assistant", "content": completion})
        messages.append({"role": "user", "content": obs.message or "Action completed."})
    return {
        "prompts": prompts,
        "completions": completions,
        "rewards": rewards,
        "total_reward": total_reward,
        "step_count": len(rewards),
        "valid_json_steps": valid_json_steps,
        "total_steps": len(completions),
        "json_valid_rate": valid_json_steps / max(len(completions), 1),
        "used_advanced_tool": used_advanced_tool,
        "final_verdict": final_verdict,
        "timed_out": final_verdict is None,
        "scenario_id": getattr(getattr(env, "_current_scenario", None), "scenario_id", None),
        "truth_label": getattr(getattr(env, "_current_scenario", None), "truth_label", None),
        "invalid_output_penalty": invalid_output_penalty,
        "reward_components": obs.reward_components if getattr(obs, "done", False) else None,
        "grade_result": getattr(env.state, "grade_result", None),
    }


def rollout_once(trainer: Any, env: Any, tokenizer: Any, task_level: str) -> dict[str, Any]:
    from trl.experimental.openenv import generate_rollout_completions

    obs = env.reset(task_level=task_level)
    prompt_ids: list[int] = []
    completion_ids: list[int] = []
    logprobs: list[float] = []
    rewards: list[float] = []
    valid_actions = 0
    total_actions = 0
    total_reward = 0.0
    invalid_output_penalty = 0.0
    for turn in range(MAX_TURNS):
        if getattr(obs, "done", False):
            break
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": obs_to_text(obs, turn + 1)},
        ]
        prompt_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False, enable_thinking=False)
        out = generate_rollout_completions(trainer, [prompt_text])[0]
        prompt_ids.extend(out["prompt_ids"])
        completion_ids.extend(out["completion_ids"])
        logprobs.extend(out["logprobs"])
        completion_text = out.get("text") or tokenizer.decode(out["completion_ids"], skip_special_tokens=True)
        action_dict, valid = parse_action(completion_text)
        total_actions += 1
        valid_actions += int(valid)
        if not valid:
            step_reward = -0.1
            rewards.append(step_reward)
            total_reward += step_reward
            invalid_output_penalty += step_reward
            continue
        try:
            action = InvestigateAction(**action_dict)
        except Exception:
            valid_actions = max(0, valid_actions - 1)
            step_reward = -0.1
            rewards.append(step_reward)
            total_reward += step_reward
            invalid_output_penalty += step_reward
            continue
        obs = env.step(action)
        step_reward = float(obs.reward or 0.0)
        rewards.append(step_reward)
        total_reward += step_reward
    grade = getattr(env.state, "grade_result", {}) or {}
    components = getattr(obs, "reward_components", None) or {}
    base_format_reward = float(components.get("format", valid_actions / max(total_actions, 1)))
    return {
        "prompt_ids": prompt_ids,
        "completion_ids": completion_ids,
        "logprobs": logprobs,
        "accuracy_reward": float(components.get("accuracy", grade.get("accuracy", 0.0))),
        "evidence_reward": float(components.get("evidence", grade.get("evidence_alignment", 0.0))),
        "reasoning_reward": float(components.get("reasoning", grade.get("reasoning_depth", 0.0))),
        "efficiency_reward": float(components.get("efficiency", grade.get("efficiency", 0.0))),
        "format_reward": base_format_reward + invalid_output_penalty,
        "anti_hack_reward": float(components.get("anti_hack", 0.0)),
        "total_reward": float((sum(components.values()) if components else obs.reward or 0.0) + invalid_output_penalty),
    }


def reward_accuracy(completions, **kwargs):
    rewards = kwargs.get("accuracy_reward")
    return [float(value) for value in rewards] if rewards else [0.0] * len(completions)


def reward_evidence(completions, **kwargs):
    rewards = kwargs.get("evidence_reward")
    return [float(value) for value in rewards] if rewards else [0.0] * len(completions)


def reward_reasoning(completions, **kwargs):
    rewards = kwargs.get("reasoning_reward")
    return [float(value) for value in rewards] if rewards else [0.0] * len(completions)


def reward_efficiency(completions, **kwargs):
    rewards = kwargs.get("efficiency_reward")
    return [float(value) for value in rewards] if rewards else [0.0] * len(completions)


def reward_format(completions, **kwargs):
    rewards = kwargs.get("format_reward")
    return [float(value) for value in rewards] if rewards else [0.0] * len(completions)


def reward_anti_hack(completions, **kwargs):
    rewards = kwargs.get("anti_hack_reward")
    return [float(value) for value in rewards] if rewards else [0.0] * len(completions)


def build_rollout_func(tokenizer: Any, env: Any):
    def rollout_func(prompts, trainer=None):
        batch = {key: [] for key in ["prompt_ids", "completion_ids", "logprobs", "accuracy_reward", "evidence_reward", "reasoning_reward", "efficiency_reward", "format_reward", "anti_hack_reward", "total_reward"]}
        for task_level in prompts:
            level = (task_level or "easy").strip()
            if level not in {"easy", "medium", "hard"}:
                level = "easy"
            episode = rollout_once(trainer=trainer, env=env, tokenizer=tokenizer, task_level=level)
            for key in batch:
                batch[key].append(episode[key])
        return batch
    return rollout_func


def sanity_check() -> None:
    env = SniffTestEnvironment()
    print("=" * 60)
    print("ENVIRONMENT SANITY CHECK")
    print("=" * 60)
    for level in ["easy", "medium", "hard"]:
        obs = env.reset(task_level=level)
        print(f"\n[{level.upper()}] {obs.claim[:90]}...")
        print(f"Visible sources: {len(obs.available_sources)}")
        if obs.available_sources:
            source_id = obs.available_sources[0].source_id
            obs = env.step(InvestigateAction(action_type="search", query="evidence fact-check"))
            obs = env.step(InvestigateAction(action_type="open_source", source_id=source_id))
            obs = env.step(
                InvestigateAction(
                    action_type="submit_verdict",
                    verdict="false",
                    confidence=0.4,
                    justification="Sanity check only. Placeholder verdict after a minimal three-step investigation.",
                )
            )
            print(f"Final reward: {float(obs.reward or 0.0):.4f}")
            print(f"Reward components: {obs.reward_components}")


def stage_dataset(level: str, episodes: int):
    from datasets import Dataset

    return Dataset.from_dict({"prompt": [level] * episodes})


def extract_reward_entries(log_history: list[dict[str, Any]]) -> list[float]:
    rewards: list[float] = []
    for entry in log_history:
        value = entry.get("reward")
        if value is None:
            value = entry.get("train/reward")
        if value is not None:
            rewards.append(float(value))
    return rewards


def stage_grpo_config(output_dir: Path, episodes: int):
    from trl import GRPOConfig

    kwargs = {
        "output_dir": str(output_dir),
        "num_train_epochs": 1,
        "per_device_train_batch_size": 4,
        "gradient_accumulation_steps": 4,
        "learning_rate": 5e-6,
        "warmup_ratio": 0.1,
        "logging_steps": 10,
        "save_steps": max(10, episodes // 2),
        "bf16": USE_BF16,
        "fp16": USE_FP16,
        "max_prompt_length": MAX_PROMPT_LENGTH,
        "max_completion_length": MAX_COMPLETION_LENGTH,
        "num_generations": NUM_GENERATIONS,
        "gradient_checkpointing": True,
        "gradient_checkpointing_kwargs": {"use_reentrant": False},
        "report_to": "none",
    }
    if USE_VLLM:
        kwargs["use_vllm"] = True
        kwargs["vllm_mode"] = VLLM_MODE
        kwargs["vllm_gpu_memory_utilization"] = VLLM_GPU_MEMORY_UTILIZATION
    return GRPOConfig(**kwargs)


def train_curriculum(output_dir: Path = TRAIN_OUTPUT_DIR, model_name: str = MODEL_NAME):
    from trl import GRPOTrainer

    model, tokenizer = load_unsloth_model(model_name=model_name)
    env = SniffTestEnvironment()
    rollout_func = build_rollout_func(tokenizer=tokenizer, env=env)
    output_dir.mkdir(parents=True, exist_ok=True)
    all_results: list[dict[str, Any]] = []
    for stage in CURRICULUM:
        level = stage["task_level"]
        episodes = int(stage["episodes"])
        threshold = stage["reward_threshold"]
        stage_output_dir = output_dir / f"{level}_stage"
        print(f"\n{'=' * 60}")
        print(f"CURRICULUM STAGE: {level.upper()} ({episodes} episodes)")
        print(f"{'=' * 60}")
        trainer = GRPOTrainer(
            model=model,
            processing_class=tokenizer,
            reward_funcs=[reward_accuracy, reward_evidence, reward_reasoning, reward_efficiency, reward_format, reward_anti_hack],
            train_dataset=stage_dataset(level, episodes),
            args=stage_grpo_config(stage_output_dir, episodes),
            rollout_func=rollout_func,
        )
        trainer.train()
        reward_values = extract_reward_entries(trainer.state.log_history)
        for idx, reward in enumerate(reward_values, start=1):
            all_results.append({"level": level, "episode": idx, "reward": reward})
        recent_rewards = reward_values[-10:]
        recent_avg = sum(recent_rewards) / len(recent_rewards) if recent_rewards else 0.0
        print(f"  Recent average reward: {recent_avg:.4f}")
        if threshold is not None and recent_avg < threshold:
            print(f"  Threshold {threshold:.2f} not reached. Keeping stage checkpoint anyway.")
        elif threshold is not None:
            print(f"  Threshold {threshold:.2f} reached.")
        model.save_pretrained(str(stage_output_dir))
        tokenizer.save_pretrained(str(stage_output_dir))
        print(f"  Checkpoint saved: {stage_output_dir}")
    return model, tokenizer, all_results


def save_final_model(model: Any, tokenizer: Any, output_dir: Path = TRAIN_OUTPUT_DIR) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    final_path = output_dir / "final_merged"
    adapter_path = output_dir / "lora_adapters"
    print(f"Saving final merged model to {final_path}...")
    model.save_pretrained_merged(str(final_path), tokenizer, save_method="merged_16bit")
    model.save_pretrained(str(adapter_path))
    tokenizer.save_pretrained(str(adapter_path))
    print(f"Merged model saved: {final_path}")
    print(f"LoRA adapters saved: {adapter_path}")


def write_reward_history(results: list[dict[str, Any]], output_dir: Path = TRAIN_OUTPUT_DIR) -> Path:
    return write_json(output_dir / "reward_history.json", results)


def load_checkpoint(checkpoint_dir: Path):
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
    if (checkpoint_dir / "adapter_config.json").exists():
        from peft import AutoPeftModelForCausalLM

        model = AutoPeftModelForCausalLM.from_pretrained(
            checkpoint_dir,
            torch_dtype="auto",
            device_map="auto",
            is_trainable=False,
        )
        model.eval()
        return model, tokenizer
    model = AutoModelForCausalLM.from_pretrained(checkpoint_dir, torch_dtype="auto", device_map="auto")
    model.eval()
    return model, tokenizer


def episode_summary(result: dict[str, Any]) -> dict[str, Any]:
    grade = result.get("grade_result") or {}
    return {
        "scenario_id": result.get("scenario_id"),
        "truth_label": result.get("truth_label"),
        "final_verdict": result.get("final_verdict"),
        "json_valid_rate": round(float(result.get("json_valid_rate", 0.0)), 4),
        "step_count": int(result.get("step_count", 0)),
        "used_advanced_tool": bool(result.get("used_advanced_tool", False)),
        "verdict_correct": bool(grade.get("accuracy", 0.0) == 1.0),
        "total_reward": round(float(result.get("total_reward", 0.0)), 4),
        "timed_out": bool(result.get("timed_out", False)),
    }


def aggregate_eval(label: str, episodes: list[dict[str, Any]]) -> dict[str, Any]:
    episode_count = len(episodes)
    json_valid_steps = sum(ep["valid_json_steps"] for ep in episodes)
    total_steps = sum(ep["total_steps"] for ep in episodes)
    verdict_correct = sum(1 for ep in episodes if (ep.get("grade_result") or {}).get("accuracy", 0.0) == 1.0)
    avg_steps_to_verdict = sum(ep["step_count"] for ep in episodes) / episode_count
    tool_use_rate = sum(1 for ep in episodes if ep["used_advanced_tool"]) / episode_count
    return {
        "model_label": label,
        "episodes": episode_count,
        "json_valid_rate": round(json_valid_steps / max(total_steps, 1), 4),
        "verdict_accuracy": round(verdict_correct / episode_count, 4),
        "avg_steps_to_verdict": round(avg_steps_to_verdict, 4),
        "tool_use_rate": round(tool_use_rate, 4),
        "avg_total_reward": round(sum(float(ep.get("total_reward", 0.0)) for ep in episodes) / episode_count, 4),
    }


def evaluate_model(label: str, model: Any, tokenizer: Any, dataset_path: Path, scenarios: list[dict[str, Any]]) -> dict[str, Any]:
    env = SniffTestEnvironment(dataset_path=dataset_path, enable_adversarial=False)
    episodes: list[dict[str, Any]] = []
    print(f"\nEvaluating {label} on {len(scenarios)} test scenarios...")
    for idx, scenario in enumerate(scenarios, start=1):
        print(f"  [{idx:02d}/{len(scenarios):02d}] {scenario['scenario_id']} ({scenario['difficulty']})")
        episodes.append(run_episode(model=model, tokenizer=tokenizer, task_level=scenario["difficulty"], env=env, scenario_id=scenario["scenario_id"]))
    summary = aggregate_eval(label, episodes)
    print(f"  json_valid_rate={summary['json_valid_rate']:.4f}  verdict_accuracy={summary['verdict_accuracy']:.4f}  avg_steps_to_verdict={summary['avg_steps_to_verdict']:.2f}  tool_use_rate={summary['tool_use_rate']:.4f}")
    return {"summary": summary, "episodes": [episode_summary(ep) for ep in episodes]}


def evaluate_checkpoints(dataset_path: Path = TEST_DATASET_PATH, output_path: Path = EVAL_METRICS_PATH, post_sft_dir: Path | None = SFT_OUTPUT_DIR, post_rl_dir: Path | None = FINAL_DIR) -> Path:
    scenarios = load_json(dataset_path)
    results = {"dataset_path": str(dataset_path), "scenario_count": len(scenarios), "models": []}
    base_model, base_tokenizer = load_unsloth_model(model_name=MODEL_NAME, add_lora=False)
    base_model.eval()
    results["models"].append(evaluate_model("baseline", base_model, base_tokenizer, dataset_path, scenarios))
    if post_sft_dir is not None and post_sft_dir.exists():
        model, tokenizer = load_checkpoint(post_sft_dir)
        results["models"].append(evaluate_model("post_sft", model, tokenizer, dataset_path, scenarios))
    if post_rl_dir is not None and post_rl_dir.exists():
        model, tokenizer = load_checkpoint(post_rl_dir)
        results["models"].append(evaluate_model("post_rl", model, tokenizer, dataset_path, scenarios))
    write_json(output_path, results)
    print(f"Saved test-set evaluation -> {output_path}")
    return output_path


def plot_reward_history(ax, results: list[dict[str, Any]], level: str, color: str) -> None:
    episodes = [row for row in results if row["level"] == level]
    rewards = [row["reward"] for row in episodes]
    window = 20
    rolling = [sum(rewards[max(0, i - window) : i + 1]) / min(i + 1, window) for i in range(len(rewards))]
    ax.plot(rewards, alpha=0.2, color=color)
    ax.plot(rolling, color=color, linewidth=2)
    ax.set_title(f"{level.capitalize()} Tasks", fontweight="bold")
    ax.set_xlabel("Episode")
    ax.set_ylabel("Reward")
    ax.set_ylim(0, 1)
    ax.axhline(y=rolling[0] if rolling else 0, color="gray", linestyle="--", alpha=0.5, label="Start")
    ax.legend()


def plot_eval_bars(ax, metrics: list[dict[str, Any]], metric_key: str, title: str) -> None:
    labels = [row["model_label"] for row in metrics]
    values = [row[metric_key] for row in metrics]
    colors = {"baseline": "#6b7280", "post_sft": "#2563eb", "post_rl": "#16a34a"}
    bars = ax.bar(labels, values, color=[colors.get(label, "#6b7280") for label in labels], width=0.6)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Model")
    ax.set_ylabel(metric_key.replace("_", " "))
    if metric_key in {"json_valid_rate", "verdict_accuracy", "tool_use_rate"}:
        ax.set_ylim(0, 1)
    else:
        upper = max(values) if values else 1
        ax.set_ylim(0, max(upper * 1.2, 1))
    for bar, value in zip(bars, values):
        label = f"{value:.3f}" if isinstance(value, float) else str(value)
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), label, ha="center", va="bottom", fontsize=9)


def maybe_eval_summaries(eval_results: dict[str, Any]) -> list[dict[str, Any]]:
    summaries: list[dict[str, Any]] = []
    for model in eval_results.get("models", []):
        summary = model.get("summary")
        if isinstance(summary, dict):
            summaries.append(summary)
    return summaries


def build_report(reward_input: Path = REWARD_HISTORY_PATH, eval_input: Path | None = EVAL_METRICS_PATH, output_path: Path = REPORT_IMAGE_PATH) -> Path:
    import matplotlib.pyplot as plt

    rewards = load_json(reward_input)
    eval_summaries = maybe_eval_summaries(load_json(eval_input)) if eval_input and eval_input.exists() else []
    fig = plt.figure(figsize=(16, 9))
    grid = fig.add_gridspec(2, 4, height_ratios=[1.4, 1.0])
    reward_axes = [fig.add_subplot(grid[0, idx]) for idx in range(3)]
    colors = {"easy": "#22c55e", "medium": "#f59e0b", "hard": "#ef4444"}
    for ax, level in zip(reward_axes, ["easy", "medium", "hard"]):
        plot_reward_history(ax, rewards, level, colors[level])
    summary_ax = fig.add_subplot(grid[0, 3])
    summary_ax.axis("off")
    total_episodes = len(rewards)
    avg_reward = sum(row["reward"] for row in rewards) / max(total_episodes, 1)
    by_level = {level: len([row for row in rewards if row["level"] == level]) for level in ["easy", "medium", "hard"]}
    summary_text = ["Training Summary", f"Episodes: {total_episodes}", f"Avg reward: {avg_reward:.3f}", f"Easy episodes: {by_level['easy']}", f"Medium episodes: {by_level['medium']}", f"Hard episodes: {by_level['hard']}"]
    if eval_summaries:
        summary_text.extend(["", "Test-set eval loaded:", f"Models: {', '.join(row['model_label'] for row in eval_summaries)}"])
    summary_ax.text(0.0, 1.0, "\n".join(summary_text), va="top", fontsize=11)
    eval_metric_specs = [("json_valid_rate", "JSON Valid Rate"), ("verdict_accuracy", "Verdict Accuracy"), ("avg_steps_to_verdict", "Avg Steps To Verdict"), ("tool_use_rate", "Tool Use Rate")]
    eval_axes = [fig.add_subplot(grid[1, idx]) for idx in range(4)]
    for ax, (metric_key, title) in zip(eval_axes, eval_metric_specs):
        if eval_summaries:
            plot_eval_bars(ax, eval_summaries, metric_key, title)
        else:
            ax.axis("off")
            ax.text(0.5, 0.5, "No test-set eval file provided", ha="center", va="center")
    fig.suptitle("SniffTest Training Curve And Fixed Test-Set Evaluation", fontsize=16, fontweight="bold")
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {output_path}")
    return output_path


def show_eval_table(path: Path = EVAL_METRICS_PATH) -> None:
    if not path.exists():
        print(f"Missing eval metrics: {path}")
        return
    payload = load_json(path)
    rows = []
    for model in payload.get("models", []):
        summary = model.get("summary", {})
        rows.append(
            f"<tr><td>{summary.get('model_label', '')}</td><td>{summary.get('episodes', '')}</td><td>{summary.get('json_valid_rate', '')}</td><td>{summary.get('verdict_accuracy', '')}</td><td>{summary.get('avg_steps_to_verdict', '')}</td><td>{summary.get('tool_use_rate', '')}</td><td>{summary.get('avg_total_reward', '')}</td></tr>"
        )
    html = "<table><thead><tr><th>model</th><th>episodes</th><th>json_valid_rate</th><th>verdict_accuracy</th><th>avg_steps_to_verdict</th><th>tool_use_rate</th><th>avg_total_reward</th></tr></thead><tbody>" + "".join(rows) + "</tbody></table>"
    display(HTML(html))


def show_reward_summary(path: Path = REWARD_HISTORY_PATH) -> None:
    if not path.exists():
        print(f"Missing reward history: {path}")
        return
    history = load_json(path)
    print(f"Episodes logged: {len(history)}")
    by_level: dict[str, list[float]] = {}
    for row in history:
        by_level.setdefault(row["level"], []).append(row["reward"])
    for level, rewards in by_level.items():
        avg_reward = sum(rewards) / max(len(rewards), 1)
        print(f"{level}: n={len(rewards)} avg_reward={avg_reward:.4f} last_reward={rewards[-1]:.4f}")


## 1. Optional Local Checks

Run these before using GPU time if you want a quick validation pass.


In [ ]:
sanity_check()


## 2. SFT Data

By default the notebook uses the committed `data/sft_trajectories.jsonl`. Set `REGENERATE_SFT_DATA=1` and provide `OPENAI_API_KEY` if you want to rebuild trajectories from `data/sft_scenarios.json`.


In [ ]:
if REGENERATE_SFT_DATA:
    generate_sft_trajectories()
else:
    print(f"Using committed SFT trajectories: {SFT_TRAJECTORIES_PATH}")

export_sft_examples()


## 3. SFT Warm Start


In [ ]:
sft_model, sft_tokenizer, sft_trainer = train_sft(output_dir=SFT_OUTPUT_DIR, model_name=MODEL_NAME)


## 4. GRPO Curriculum Training


In [ ]:
# USE_VLLM = False
print("Using GRPO without vLLM for compatibility with notebook runtimes.")
rl_model, rl_tokenizer, reward_history = train_curriculum(output_dir=TRAIN_OUTPUT_DIR, model_name=str(SFT_OUTPUT_DIR))
write_reward_history(reward_history, TRAIN_OUTPUT_DIR)
save_final_model(rl_model, rl_tokenizer, TRAIN_OUTPUT_DIR)


## 5. Fixed Test-Set Evaluation And Report


In [ ]:
evaluate_checkpoints(dataset_path=TEST_DATASET_PATH, output_path=EVAL_METRICS_PATH, post_sft_dir=SFT_OUTPUT_DIR, post_rl_dir=FINAL_DIR)
build_report(REWARD_HISTORY_PATH, EVAL_METRICS_PATH, REPORT_IMAGE_PATH)


## 6. Artifact Summary


In [ ]:
for path in [
    SFT_TRAJECTORIES_PATH,
    SFT_CHAT_PATH,
    SFT_OUTPUT_DIR,
    TRAIN_OUTPUT_DIR,
    FINAL_DIR,
    ADAPTER_DIR,
    REWARD_HISTORY_PATH,
    EVAL_METRICS_PATH,
    REPORT_IMAGE_PATH,
]:
    print(f"{path}: {'present' if path.exists() else 'missing'}")

show_reward_summary(REWARD_HISTORY_PATH)
show_eval_table(EVAL_METRICS_PATH)
if REPORT_IMAGE_PATH.exists():
    display(Image(filename=str(REPORT_IMAGE_PATH)))
else:
    print(f"Missing report image: {REPORT_IMAGE_PATH}")
